# 🏥 AMOS22 L3 VFA/PMA Pipeline - TS Path FIX

**BUG FIX**: case_id parsing düzeltildi (.nii uzantısı kaldırıldı)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/L3_SO_ANALYSIS')

In [ ]:
!pip install SimpleITK nibabel scikit-image opencv-python-headless scipy tqdm pandas torch torchvision monai -q

import numpy as np
import SimpleITK as sitk
import cv2
from scipy import ndimage
from skimage import morphology, measure, segmentation
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from monai.losses import DiceLoss
from monai.networks.nets import UNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
AMOS_ROOT = Path("/content/drive/MyDrive/AMOS221/imagesTr")
TS_ROOT = Path("/content/drive/MyDrive/TS_teachers_AMOS22")
OUTPUT_ROOT = Path("/content/drive/MyDrive/L3_SO_ANALYSIS/amos_results")
MODEL_DIR = OUTPUT_ROOT / "dl_models"

OUTPUT_ROOT.mkdir(exist_ok=True, parents=True)
MODEL_DIR.mkdir(exist_ok=True, parents=True)

amos_cases = sorted(AMOS_ROOT.glob("amos_*.nii.gz"))
print(f"AMOS cases: {len(amos_cases)}")

In [ ]:
def load_nifti_volume(path):
    img = sitk.ReadImage(str(path))
    volume = sitk.GetArrayFromImage(img)
    spacing = img.GetSpacing()
    return volume, spacing, img

def get_pixel_area_mm2(spacing):
    return spacing[0] * spacing[1]

HU_FAT_LOW, HU_FAT_HIGH = -190, -30
HU_MUSCLE_LOW, HU_MUSCLE_HIGH = -29, 150

In [ ]:
def detect_vertebra_center(hu_slice, hu_threshold=150):
    bone_mask = hu_slice > hu_threshold
    bone_mask = morphology.remove_small_objects(bone_mask, min_size=50)
    bone_mask = morphology.binary_closing(bone_mask, morphology.disk(3))
    labels = measure.label(bone_mask)
    regions = measure.regionprops(labels)
    if len(regions) == 0:
        return None, 0.0
    h, w = hu_slice.shape
    center_x = w // 2
    best_region, best_score = None, -1
    for region in regions:
        y, x = region.centroid
        if y > h * 0.6 or abs(x - center_x) / w > 0.2:
            continue
        if region.area < 100 or region.area > 2500:
            continue
        score = region.area * region.solidity * (1 - abs(x - center_x) / w)
        if score > best_score:
            best_score, best_region = score, region
    if best_region is None:
        return None, 0.0
    return best_region.centroid, min(best_score / 1000.0, 1.0)

def find_l3_slice_index(volume, start_frac=0.5, search_range=30):
    D, H, W = volume.shape
    start_z = int(D * start_frac)
    best_z, best_conf = start_z, 0.0
    for z in range(max(0, start_z - search_range), min(D, start_z + search_range)):
        center, conf = detect_vertebra_center(volume[z, :, :])
        if center is not None and conf > best_conf:
            best_conf, best_z = conf, z
    return best_z, best_conf

In [ ]:
def create_body_mask(hu_slice, air_threshold=-900):
    body = hu_slice > air_threshold
    labels = measure.label(body)
    if labels.max() == 0:
        return body
    regions = measure.regionprops(labels)
    largest = max(regions, key=lambda r: r.area)
    return labels == largest.label

def create_wall_mask(seg_slice):
    wall = seg_slice > 0
    wall = morphology.binary_dilation(wall, morphology.disk(2))
    return wall

def compute_inner_abdomen_mask(hu_slice, seg_slice, vertebra_center=None):
    body_mask = create_body_mask(hu_slice)
    wall_mask = create_wall_mask(seg_slice)
    if vertebra_center is None:
        vertebra_center, _ = detect_vertebra_center(hu_slice)
    if vertebra_center is None:
        h, w = hu_slice.shape
        seed_y, seed_x = h // 2, w // 2
    else:
        seed_y, seed_x = int(vertebra_center[0]), int(vertebra_center[1])
    fillable = body_mask & (~wall_mask)
    inner_abdomen = np.zeros_like(fillable, dtype=bool)
    if fillable[seed_y, seed_x]:
        inner_abdomen = segmentation.flood(fillable.astype(np.uint8), (seed_y, seed_x), connectivity=2)
    else:
        dist = ndimage.distance_transform_edt(fillable)
        if dist.max() > 0:
            max_pos = np.unravel_index(dist.argmax(), dist.shape)
            inner_abdomen = segmentation.flood(fillable.astype(np.uint8), max_pos, connectivity=2)
    return inner_abdomen.astype(bool)

In [ ]:
def compute_vfa(hu_slice, inner_mask, pixel_area_mm2):
    fat_mask = (hu_slice >= HU_FAT_LOW) & (hu_slice <= HU_FAT_HIGH)
    vfa_mask = fat_mask & inner_mask
    vfa_pixels = vfa_mask.sum()
    vfa_mm2 = vfa_pixels * pixel_area_mm2
    return {'vfa_mask': vfa_mask, 'vfa_mm2': float(vfa_mm2), 'vfa_cm2': float(vfa_mm2 / 100)}

def estimate_psoas_mask(hu_slice, vertebra_center=None):
    if vertebra_center is None:
        vertebra_center, _ = detect_vertebra_center(hu_slice)
    if vertebra_center is None:
        return np.zeros_like(hu_slice, dtype=bool)
    muscle_mask = (hu_slice >= HU_MUSCLE_LOW) & (hu_slice <= HU_MUSCLE_HIGH)
    vy, vx = int(vertebra_center[0]), int(vertebra_center[1])
    h, w = hu_slice.shape
    roi_mask = np.zeros_like(hu_slice, dtype=bool)
    for side_sign in [-1, 1]:
        px, py = vx + side_sign * 60, vy + 30
        y1, y2 = max(0, py - 40), min(h, py + 40)
        x1, x2 = max(0, px - 40), min(w, px + 40)
        roi_mask[y1:y2, x1:x2] = True
    psoas_mask = muscle_mask & roi_mask
    psoas_mask = morphology.remove_small_objects(psoas_mask, min_size=50)
    return morphology.binary_closing(psoas_mask, morphology.disk(3))

def compute_pma(hu_slice, psoas_mask, pixel_area_mm2, height_m=1.7):
    pma_pixels = psoas_mask.sum()
    pma_mm2 = pma_pixels * pixel_area_mm2
    pma_cm2 = pma_mm2 / 100
    pmi = pma_cm2 / (height_m ** 2)
    return {'psoas_mask': psoas_mask, 'pma_mm2': float(pma_mm2), 'pma_cm2': float(pma_cm2), 'pmi': float(pmi)}

---
# DEEP LEARNING: Dataset
---

In [ ]:
class AMOSVFAPMA_Dataset(Dataset):
    def __init__(self, cases_data):
        self.cases = cases_data
    def __len__(self):
        return len(self.cases)
    def __getitem__(self, idx):
        case = self.cases[idx]
        hu_norm = np.clip((case['hu_slice'] + 150) / 400, 0, 1).astype(np.float32)
        ts_norm = (case['ts_seg'] > 0).astype(np.float32)
        vb_norm = case['vertebra_mask'].astype(np.float32)
        input_tensor = np.stack([hu_norm, ts_norm, vb_norm], axis=0)
        inner_mask = case['inner_abdomen_mask'].astype(np.float32)
        vfa_mask = case['vfa_mask'].astype(np.float32)
        pma_mask = case['pma_mask'].astype(np.float32)
        target_tensor = np.stack([vfa_mask, pma_mask, inner_mask], axis=0)
        return {'input': torch.from_numpy(input_tensor), 'target': torch.from_numpy(target_tensor), 'case_id': case['case_id']}

In [ ]:
# 🔧 BUG FIX: case_id parsing düzeltildi
def prepare_training_data(amos_cases, ts_root, max_cases=50):
    training_data = []
    if max_cases:
        amos_cases = amos_cases[:max_cases]
    print(f"📦 Processing {len(amos_cases)} cases...")
    for amos_path in tqdm(amos_cases):
        # FIX: .nii.gz uzantısını kaldır
        case_id = amos_path.name.replace('.nii.gz', '')
        print(f"\n🔍 {case_id}: Processing...")
        try:
            hu_vol, spacing, _ = load_nifti_volume(amos_path)
            # TS path FIX: .nii uzantısı YOK
            ts_seg_path = ts_root / case_id / "abdominal_muscles.nii.gz"
            print(f"  TS path: {ts_seg_path}")
            if not ts_seg_path.exists():
                print(f"  ⚠️ TS segment bulunamadı")
                continue
            seg_vol, _, _ = load_nifti_volume(ts_seg_path)
            z_l3, vb_conf = find_l3_slice_index(hu_vol)
            if vb_conf < 0.5:
                print(f"  ⚠️ Low VB conf: {vb_conf:.2f}")
                continue
            hu_l3, seg_l3 = hu_vol[z_l3, :, :], seg_vol[z_l3, :, :]
            vb_center, _ = detect_vertebra_center(hu_l3)
            if vb_center is None:
                print(f"  ⚠️ VB center not found")
                continue
            vb_mask = (hu_l3 > 150).astype(bool)
            vb_mask = morphology.remove_small_objects(vb_mask, min_size=50)
            inner_mask = compute_inner_abdomen_mask(hu_l3, seg_l3, vb_center)
            pixel_area = get_pixel_area_mm2(spacing)
            vfa_result = compute_vfa(hu_l3, inner_mask, pixel_area)
            psoas_mask = estimate_psoas_mask(hu_l3, vb_center)
            # Resize to 512x512
            target_size = (512, 512)
            hu_l3 = cv2.resize(hu_l3, target_size, interpolation=cv2.INTER_LINEAR)
            seg_l3 = cv2.resize(seg_l3.astype(np.float32), target_size, interpolation=cv2.INTER_NEAREST)
            vb_mask = cv2.resize(vb_mask.astype(np.float32), target_size, interpolation=cv2.INTER_NEAREST).astype(bool)
            inner_mask = cv2.resize(inner_mask.astype(np.float32), target_size, interpolation=cv2.INTER_NEAREST).astype(bool)
            vfa_mask = cv2.resize(vfa_result['vfa_mask'].astype(np.float32), target_size, interpolation=cv2.INTER_NEAREST).astype(bool)
            psoas_mask = cv2.resize(psoas_mask.astype(np.float32), target_size, interpolation=cv2.INTER_NEAREST).astype(bool)
            training_data.append({'case_id': case_id, 'hu_slice': hu_l3, 'ts_seg': seg_l3, 'vertebra_mask': vb_mask, 'inner_abdomen_mask': inner_mask, 'vfa_mask': vfa_mask, 'pma_mask': psoas_mask})
            print(f"  ✅ Added to training data")
        except Exception as e:
            print(f"  ❌ Error: {e}")
            continue
    print(f"\n✅ {len(training_data)} cases ready")
    return training_data

training_data = prepare_training_data(amos_cases, TS_ROOT, max_cases=50)

In [ ]:
full_dataset = AMOSVFAPMA_Dataset(training_data)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

In [ ]:
model = UNet(spatial_dims=2, in_channels=3, out_channels=3, channels=(32, 64, 128, 256, 512), strides=(2, 2, 2, 2), num_res_units=2, dropout=0.1).to(device)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
dice_loss = DiceLoss(sigmoid=True)
bce_loss = nn.BCEWithLogitsLoss()
def combined_loss(pred, target):
    return dice_loss(pred, target) + bce_loss(pred, target)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60, eta_min=1e-6)

In [ ]:
from torch.cuda.amp import autocast, GradScaler
num_epochs = 60
scaler = GradScaler()
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'lr': []}
best_model_path = MODEL_DIR / "best_unet.pt"
for epoch in range(num_epochs):
    model.train()
    train_loss_epoch = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        inputs, targets = batch['input'].to(device), batch['target'].to(device)
        optimizer.zero_grad()
        with autocast():
            outputs = model(inputs)
            loss = combined_loss(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss_epoch += loss.item()
    train_loss_avg = train_loss_epoch / len(train_loader)
    model.eval()
    val_loss_epoch = 0
    with torch.no_grad():
        for batch in val_loader:
            inputs, targets = batch['input'].to(device), batch['target'].to(device)
            with autocast():
                outputs = model(inputs)
                loss = combined_loss(outputs, targets)
            val_loss_epoch += loss.item()
    val_loss_avg = val_loss_epoch / len(val_loader)
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss_avg)
    history['val_loss'].append(val_loss_avg)
    history['lr'].append(current_lr)
    if val_loss_avg < best_val_loss:
        best_val_loss = val_loss_avg
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'train_loss': train_loss_avg, 'val_loss': val_loss_avg}, best_model_path)
        print(f"💾 Best: {val_loss_avg:.4f}")
    print(f"Epoch {epoch+1}: Train={train_loss_avg:.4f} Val={val_loss_avg:.4f} LR={current_lr:.6f}")
print(f"✅ Done! Best val: {best_val_loss:.4f}")

# ✅ Training Complete!